# 02 — Feature Engineering

**Objetivo:** Limpar duplicatas, fazer o split estratificado (preservando os 0,17% de fraude em treino e teste) e montar o preprocessor (RobustScaler no Amount, V's passthrough, Time descartado), sem aplicar balanceamento no split para evitar leakage.

---

**Roteiro:**

1. Setup
2. Carregamento dos dados
3. Limpeza de duplicatas
4. Split treino/teste estratificado
5. Pré-processamento
6. Salvar artefatos


### Etapa 1 - Setup

In [1]:
%load_ext autoreload
%autoreload 2

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler


from src.config import CONFIG, CAMINHOS
from src.viz_config import aplicar_tema_seaborn, PALETTE, CORES

DADOS_BRUTOS = CAMINHOS.dados_raw / CAMINHOS.dados_brutos
DADOS_PRO    = CAMINHOS.dados_processed
MODELS_DIR   = CAMINHOS.modelos
RANDOM_STATE = CONFIG["dados"]["random_state"]
TARGET       = CONFIG["dados"]["target_col"]
TEST_SIZE    = CONFIG["dados"]["test_size"]

DADOS_PRO.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

aplicar_tema_seaborn()

print("\n✅ Setup configurado!")

print(f"SEED: {RANDOM_STATE}  |  TARGET: {TARGET}  |  TEST_SIZE: {TEST_SIZE}\n")
print(f"MODELOS          : {MODELS_DIR}")
print(f"DADOS BRUTOS     : {DADOS_BRUTOS}")
print(f"DADOS PROCESSADOS: {DADOS_PRO}")


✅ Setup configurado!
SEED: 42  |  TARGET: Class  |  TEST_SIZE: 0.2

MODELOS          : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\models
DADOS BRUTOS     : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\data\raw\creditcard.zip
DADOS PROCESSADOS: C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\data\processed


### Etapa 2 - Carregamento dos dados

In [2]:
df = pd.read_csv(DADOS_BRUTOS)
print(f"✅ Carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"   Fraudes: {(df[TARGET]==1).sum()} ({(df[TARGET]==1).mean()*100:.3f}%)")

df.head()

✅ Carregado: 284,807 linhas × 31 colunas
   Fraudes: 492 (0.173%)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


### Etapa 3 - Limpeza de duplicatas

Durante a inspeção inicial realizada no NB01, foram identificados registros duplicados no dataset. Nesta etapa, essas duplicatas serão tratadas para evitar impactos negativos nas análises e na modelagem.

In [3]:
n_antes = len(df)
n_dup   = df.duplicated().sum()
print(f"Duplicatas encontradas: {n_dup:,}")

Duplicatas encontradas: 1,081


In [4]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"✅ Após remover: {len(df):,} linhas (-{n_antes-len(df):,})")

✅ Após remover: 283,726 linhas (-1,081)


In [5]:
print("═" * 60)
print("Verificação")
print("═" * 60)

print(f"  Fraudes restantes: {(df[TARGET]==1).sum()} ({(df[TARGET]==1).mean()*100:.3f}%)")
print(f"  Legítimas        : {(df[TARGET]==0).sum():,}")

════════════════════════════════════════════════════════════
Verificação
════════════════════════════════════════════════════════════
  Fraudes restantes: 473 (0.167%)
  Legítimas        : 283,253


Após a remoção dos registros duplicados, o número de transações fraudulentas reduziu de **492 para 473**, indicando que **19 fraudes estavam duplicadas** no dataset.

Essa etapa tornou o desbalanceamento da variável alvo ainda mais acentuado:

* Antes da remoção: **0,173%** de fraudes
* Após a remoção: **0,167%** de fraudes

Diante desse cenário, o uso de um **split treino/teste estratificado** torna-se obrigatório. Sem estratificação, a divisão aleatória poderia distribuir as poucas fraudes de forma inadequada, concentrando casos positivos majoritariamente no treino ou no teste.

A estratificação garante que ambos os conjuntos preservem a proporção original das classes, permitindo uma avaliação mais confiável do modelo em um problema de classificação extremamente desbalanceado.

### Etapa 4 - Split treino/teste estratificado

Nesta etapa, será realizado o **split treino/teste estratificado**, utilizando o parâmetro `stratify=y`.Essa estratégia garante que a proporção de fraudes seja preservada tanto no conjunto de treino quanto no conjunto de teste, o que é essencial em um problema com **desbalanceamento extremo** da classe alvo.

O balanceamento das classes **não será aplicado antes do split**. Essa decisão evita **data leakage**, pois técnicas de balanceamento, como undersampling, oversampling ou SMOTE, devem ser aplicadas apenas nos dados de treino e, preferencialmente, dentro do pipeline de modelagem.

In [6]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

In [7]:
print("═" * 60)
print("SPLIT ESTRATIFICADO")
print("═" * 60)

print(f"  Treino: {len(X_train):,} linhas · {y_train.sum()} fraudes ({y_train.mean()*100:.3f}%)")
print(f"  Teste : {len(X_test):,} linhas ·  {y_test.sum()} fraudes ({y_test.mean()*100:.3f}%)")
print(f"\n  ✅ Proporção de fraude preservada nos dois conjuntos")

════════════════════════════════════════════════════════════
SPLIT ESTRATIFICADO
════════════════════════════════════════════════════════════
  Treino: 226,980 linhas · 378 fraudes (0.167%)
  Teste : 56,746 linhas ·  95 fraudes (0.167%)

  ✅ Proporção de fraude preservada nos dois conjuntos


### Etapa 5 - Pré-processamento

Nesta etapa, será definido o **pré-processador** do projeto utilizando uma estrutura baseada em `Pipeline` e `ColumnTransformer`.

As colunas serão tratadas de acordo com suas características:

* **V1 a V28**: variáveis anonimizadas provenientes de PCA. Como já estão transformadas, serão mantidas sem novas transformações adicionais.
* **Time**: representa o tempo absoluto em segundos desde a primeira transação do dataset. Por não ser uma variável generalizável para novos dados, será descartada nesta etapa.
* **Amount**: apresenta forte assimetria e presença de outliers, conforme observado na EDA. Por isso, será escalada com **RobustScaler**, método mais adequado para variáveis com valores extremos.

In [8]:
V_COLS = [c for c in X.columns if c.startswith("V")]
SCALE_COLS  = ["Amount"]
DROP_COLS   = ["Time"] 

print(f"V's (passthrough)  : {len(V_COLS)} colunas")
print(f"Escalar (Robust)   : {SCALE_COLS}")
print(f"Descartar          : {DROP_COLS}")

V's (passthrough)  : 28 colunas
Escalar (Robust)   : ['Amount']
Descartar          : ['Time']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("amount", RobustScaler(), SCALE_COLS),
        ("vs", "passthrough", V_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('amount', ...), ('vs', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g.

Após definir o pré-processador, o ajuste será realizado **somente no conjunto de treino**.

Em seguida, o pré-processador será utilizado para transformar os dados e permitir a inspeção da base processada: `fit_transform` aplicado em `X_train`.

In [10]:
X_train_proc = preprocessor.fit_transform(X_train)

print(f"\n✅ Preprocessor fitado")
print(f"   Shape transformado: {X_train_proc.shape} (29 features: 28 V's + Amount)")
print(f"   Time descartado, Amount escalado, V's preservadas")

print(f"\nAmount após RobustScaler:")
print("─" * 30)
X_train_proc["Amount"].describe().round(3)


✅ Preprocessor fitado
   Shape transformado: (226980, 29) (29 features: 28 V's + Amount)
   Time descartado, Amount escalado, V's preservadas

Amount após RobustScaler:
──────────────────────────────


count    226980.000
mean          0.919
std           3.406
min          -0.306
25%          -0.227
50%           0.000
75%           0.773
max         272.096
Name: Amount, dtype: float64

### Etapa 6 - Salvar artefatos

Após a etapa de pré-processamento, os conjuntos de treino e teste serão salvos em formato Parquet.

In [11]:
X_train.to_parquet(DADOS_PRO / "X_train.parquet", index=False)
X_test.to_parquet(DADOS_PRO  / "X_test.parquet",  index=False)

y_train.to_frame().to_parquet(DADOS_PRO / "y_train.parquet", index=False)
y_test.to_frame().to_parquet(DADOS_PRO  / "y_test.parquet",  index=False)

print(f"X_train.parquet  {X_train.shape}")
print(f"X_test.parquet   {X_test.shape}")
print(f"y_train.parquet  ({len(y_train)},) · {y_train.sum()} fraudes")
print(f"y_test.parquet   ({len(y_test)},) · {y_test.sum()} fraudes")

X_train.parquet  (226980, 30)
X_test.parquet   (56746, 30)
y_train.parquet  (226980,) · 378 fraudes
y_test.parquet   (56746,) · 95 fraudes


O objeto preprocessor também será salvo como artefato de referência do projeto.

In [12]:
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.pkl")

print(f"preprocessor.pkl (RobustScaler Amount + passthrough V's)")

preprocessor.pkl (RobustScaler Amount + passthrough V's)


In [13]:
print("═" * 60)
print("Round-trip")
print("═" * 60)

chk = pd.read_parquet(DADOS_PRO / "X_train.parquet")
print(f" {'✅' if chk.shape==X_train.shape else '❌'} round-trip X_train ok")
print(f" {'✅' if 'Time' in chk.columns else '⚠️'} Time ainda nos dados crus "
      f"(será descartado pelo preprocessor no pipeline)")

════════════════════════════════════════════════════════════
Round-trip
════════════════════════════════════════════════════════════
 ✅ round-trip X_train ok
 ✅ Time ainda nos dados crus (será descartado pelo preprocessor no pipeline)


### Etapa 7 - Fechamento

**LIMPEZA**
* 1.081 duplicatas removidas → 283.726 linhas
* Fraudes: 492 → 473 (0.167%, razão ~599:1)

**SPLIT ESTRATIFICADO (stratify obrigatório com classe rara)**
* Treino: 226.980 · 378 fraudes
* Teste : 56.746 · 95 fraudes
* Balanceamento NÃO aplicado aqui — vai no pipeline de CV (NB03)
    para evitar data leakage

**PREPROCESSOR (enxuto — V's já são PCA)**
* Amount → RobustScaler (robusto aos outliers até £25k)
* V1-V28 → passthrough (já centradas pelo PCA)
* Time → descartado (valor absoluto, não generaliza)
* 29 features finais

**ARTEFATOS**
* X_train/X_test/y_train/y_test.parquet · preprocessor.pkl

**PRÓXIMO PASSO → NB03 (Modelagem Baseline)**
* DECIDIR balanceamento: class_weight vs SMOTE vs combinação
* Comparar modelos por PR-AUC (StratifiedKFold)
* ImbPipeline para balanceamento dentro do CV (sem leakage)